# GR00T N1.7 on an OXE dataset: dataset info and the model adapter

`robotdataset` returns TED batches: flat TensorDicts keyed by `"/"`-separated paths. It does not know which leaf
is which model input, in what order, or where a group sits inside an assembled state or action vector. Two layers
close that gap without touching any dataset metadata files:

```
loader ──▶ DatasetInfo ──▶ viola_gr00t.yaml ──(you edit it)──▶ Gr00tAdapter ──▶ VLAStepData, GR00T configs, statistics
          (robotdataset.info)                                  (robotdataset.adapters.gr00t)
```

This notebook walks that path for the Open X-Embodiment dataset `viola` and stops at the model inputs of
`Gr00tN1d7Processor`; it needs neither GPU nor model weights. The training loop that consumes the same objects is
`finetune_scratch.ipynb` in the `finetune-groot` repository (reference docs: `doc/dataset_info.md`, `doc/adapters.md`).

Requirements: `robotdataset[oxe]` and the `gr00t` package (Isaac-GR00T N1.7). The first run streams five episodes
from `gs://gresearch/robotics/viola` into the robotdataset cache; later runs are offline.

In [2]:
from robotdataset import list_datasets
list_datasets()

{'asimov_dilemmas_scifi_train': ['0.1.0'],
 'asimov_dilemmas_auto_val': ['0.1.0'],
 'agent_aware_affordances': ['1.0.0'],
 'asimov_v2_constraints_without_rationale': ['0.1.0'],
 'asimov_v2_constraints_with_rationale': ['0.1.0'],
 'asimov_v2_injuries': ['0.1.0'],
 'asu_table_top_converted_externally_to_rlds': ['0.1.0'],
 'austin_buds_dataset_converted_externally_to_rlds': ['0.1.0'],
 'asimov_v2_videos': ['0.1.0'],
 'asimov_multimodal_manual_val': ['0.1.0'],
 'asimov_dilemmas_scifi_val': ['0.1.0'],
 'aloha_mobile': ['0.0.1'],
 'berkeley_cable_routing': ['0.1.0'],
 'asimov_multimodal_auto_val': ['0.1.0'],
 'berkeley_fanuc_manipulation': ['0.1.0'],
 'austin_sirius_dataset_converted_externally_to_rlds': ['0.1.0'],
 'asimov_multimodal_robopair_val': ['0.1.0'],
 'berkeley_autolab_ur5': ['0.1.0'],
 'austin_sailor_dataset_converted_externally_to_rlds': ['0.1.0'],
 'berkeley_gnm_cory_hall': ['0.1.0'],
 'asimov_injury_val': ['0.1.0'],
 'berkeley_gnm_sac_son': ['0.1.0'],
 'berkeley_rpt_converted_e

In [1]:
from dataclasses import replace
from pathlib import Path

import numpy as np

from robotdataset.adapters.gr00t import Gr00tAdapter
from robotdataset.info import DatasetInfo

TAG = "new_embodiment"  # GR00T's embodiment tag for a custom robot
TRAIN_EPISODES, VAL_EPISODES = [0, 1, 2], [4]
YAML_PATH = Path("dataset_taco_play_gr00t.yaml")

/work/yn030245/venv/finetune-groot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1791982804.253933    5104 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791982804.305947    5104 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791982809.454006    5104 port.cc:153] oneDNN custom operations

In [2]:
YAML_PATH

PosixPath('dataset_taco_play_gr00t.yaml')

## 1. List what the dataset provides

`DatasetInfo.from_loader` builds the loader on a few episodes and describes every TED leaf: path, kind, dtype and
per-step shape (images are HWC as stored). TED bookkeeping (`next/...`, `done`, ...) is hidden. The same listing is
available from a shell: `python -m robotdataset.info --loader OXEDataset --kwargs '{"dataset_name": "viola"}' --episodes 0:4 --adapter gr00t -o viola_gr00t.yaml`.

In [4]:
info = DatasetInfo.from_loader("LeRobotVideoDataset", {"repo_id": "LSY-lab/stack_without_ft_tact_v4"}, episodes=TRAIN_EPISODES)
info.to_yaml(YAML_PATH)
print(info.table())

Assembling training buffer: 100%|██████████| 3/3 [00:04<00:00,  1.52s/ep]

LeRobotVideoDataset: 3 episode(s), 725 step(s), 15 fps
path                              kind    dtype    shape          use  name
--------------------------------  ------  -------  -------------  ---  --------------------
observation/images/primary        image   uint8    [256, 256, 3]  no   primary
observation/images/wrist          image   uint8    [256, 256, 3]  no   wrist
observation/state/cartesian       state   float32  [6]            no   cartesian
observation/state/gripper         state   float32  [1]            no   gripper
observation/state/joints          state   float32  [7]            no   joints
observation/state/target          state   float32  [6]            no   target
observation/state/all             state   float32  [20]           no   all
action                            action  float32  [7]            no   action
observation/language_instruction  text    string   []             no   language_instruction


In [11]:
info = DatasetInfo.from_loader("OXEDataset", {"dataset_name": "taco_play"}, episodes=TRAIN_EPISODES)
info.to_yaml(YAML_PATH)
print(info.table())

OXEDataset: 4 episode(s), 264 step(s)
path                                         kind    dtype    shape          use  name
-------------------------------------------  ------  -------  -------------  ---  -------------------------------
observation/depth_static                     image   float32  [150, 200]     no   depth_static
observation/rgb_static                       image   uint8    [150, 200, 3]  no   rgb_static
observation/rgb_gripper                      image   uint8    [84, 84, 3]    no   rgb_gripper
observation/depth_gripper                    image   float32  [84, 84]       no   depth_gripper
observation/robot_obs                        state   float32  [15]           no   robot_obs
action/rel_actions_gripper                   action  float32  [7]            no   rel_actions_gripper
action/actions                               action  float32  [7]            no   actions
action/rel_actions_world                     action  float32  [7]            no   rel_actions_world

In [12]:
info = DatasetInfo.from_yaml(YAML_PATH)
info = Gr00tAdapter.with_defaults(info)  # adds the gr00t: block with its defaults (horizon 16, windows [0])
info.to_yaml(YAML_PATH)

'# robotdataset dataset info (generated). Edit it, then load it with a model adapter.\n#\n#   use    true on every modality the model should get (all start as false)\n#   order  the order of this list is the order handed to the model\n#   name   the model-side group name (defaults to the shortest unique path suffix)\n#   role   optional override of the role implied by kind: video | state | action | language\n#   split  optional, cuts one vector leaf into named groups: [{name: eef, columns: [0, 6]}, ...]\n#\n# kind, dtype and shape describe the data; they are checked against the loader when the file is loaded.\ndataset:\n  loader: OXEDataset\n  kwargs: {dataset_name: taco_play}\n  episodes_loaded: 4\n  steps_loaded: 264\n\nmodalities:\n  - {use: false, path: observation/depth_static,                    kind: image,  dtype: float32, shape: [150, 200],    name: depth_static}\n  - {use: true,  path: observation/rgb_static,                      kind: image,  dtype: uint8,   shape: [150, 200

In [ ]:
adapter = Gr00tAdapter.from_yaml(str(YAML_PATH))

modality_configs = adapter.modality_configs()  # GR00T ModalityConfig per modality, ready for the processor
{k: (list(v.modality_keys), list(v.delta_indices)) for k, v in modality_configs.items()}

{'video': (['rgb_static'], [0]),
 'state': (['robot_obs'], [0]),
 'action': (['actions'],
  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]),
 'language': (['annotation.human.task_description'], [0])}

## 4. Build the loaders and look at a batch

`build_dataset` instantiates the loader named in the file with the sampling windows the selection needs (one frame
for video and state, a 16-step window for action) and checks the file against a fresh listing of the loader and a
sampled batch. A stale file fails with one message naming everything that changed. `to_steps` cuts the sampled
leaves into the selected groups and wraps each sample as GR00T's `VLAStepData`.

In [41]:
train_ds = adapter.build_dataset(episodes=TRAIN_EPISODES, batch_size=4)
val_ds = adapter.build_dataset(episodes=VAL_EPISODES, batch_size=4)
print("train:", train_ds.num_episodes, "episodes,", len(train_ds), "steps | val:", val_ds.num_episodes, "episodes,", len(val_ds), "steps")

batch = train_ds.sample()
print({m.path: tuple(batch[m.path].shape) for m in adapter.info.selected if not m.is_string})  # (B, T, ...) per TED leaf

step = adapter.to_steps(batch, TAG)[0]
print("text   :", step.text)
print("images :", {k: (np.asarray(v[0]).shape, str(np.asarray(v[0]).dtype)) for k, v in step.images.items()})
print("states :", {k: v.shape for k, v in step.states.items()})
print("actions:", {k: v.shape for k, v in step.actions.items()})

train: 4 episodes, 264 steps | val: 1 episodes, 66 steps
{'observation/rgb_static': (4, 1, 3, 150, 200), 'observation/robot_obs': (4, 1, 15), 'action/actions': (4, 16, 7)}
text   : place the pink block on top of the purple block
images : {'rgb_static': ((150, 200, 3), 'uint8')}
states : {'robot_obs': (1, 15)}
actions: {'actions': (16, 7)}


In [26]:
modality_configs["video"].modality_keys

['rgb_static']

In [33]:
train_ds.episodes

[0, 1, 2, 3]

In [37]:
from robotdataset import TemporalSampler
sampler = TemporalSampler(
    delta_timestamps={
        train_ds.image_keys[0]: np.arange(-1, 0, 1/10).tolist(),
        # "observation/finger_vision_1": np.arange(-3, 0, 1/10).tolist(),
    },
    control_frequency=10
)
train_ds.set_sampler(sampler=sampler)

In [38]:
train_ds.image_keys[0]

'observation/depth_gripper'

In [36]:
modality_configs["video"].modality_keys[0]

'rgb_static'

In [42]:
from robotdataset import episodeViz, batchViz

episodeViz(train_ds, 1, key="observation/"+modality_configs["video"].modality_keys[0], is_output_video=True)

## 5. Statistics for the processor

GR00T normalises state and action with per-group statistics. The adapter computes them from the loader you pass:
build that loader from the training episodes only, so held-out episodes do not leak into the normalisation. Nothing is
read from or written to the dataset's metadata files. Quantiles are exact by default because GR00T normalises with
q01/q99; groups declared `RELATIVE` additionally get relative-action statistics (not needed for `viola`).

In [43]:
stats = adapter.statistics(train_ds, TAG)
for role in ("state", "action"):
    for group, s in stats[TAG][role].items():
        print(f"{role:<6} {group:<28} q01 {np.round(s['q01'], 3)}  q99 {np.round(s['q99'], 3)}")

Exact quantiles: 100%|██████████| 22/22 [00:00<00:00, 9462.13dim/s]

state  robot_obs                    q01 [ 0.306 -0.183  0.203 -3.137 -0.356 -0.195  0.    -0.183 -1.008 -0.443
 -2.865 -0.301  1.556 -0.868 -1.   ]  q99 [ 0.48   0.47   0.531  3.138 -0.005  1.776  0.081  0.456  0.172  0.499
 -1.39   0.57   2.398  0.921  1.   ]
action actions                      q01 [ 0.292 -0.242  0.191 -3.139 -0.357 -0.195 -1.   ]  q99 [0.503 0.484 0.548 3.139 0.028 1.821 1.   ]


## 6. Through the GR00T N1.7 processor

The processor is built exactly as the fine-tuning notebook builds it (here with the defaults of `Gr00tN1d7Config`).
`configure_processor` hands it the statistics and `to_model_batch` goes from a sampled batch to the collated
model inputs, so `model(**inputs)` is the next step in training. Building the processor downloads the tokenizer and
image processor of the Cosmos-Reason2 backbone on first use.

In [7]:
from gr00t.configs.model.gr00t_n1d7 import Gr00tN1d7Config
from gr00t.model.gr00t_n1d7.processing_gr00t_n1d7 import Gr00tN1d7Processor

mc = Gr00tN1d7Config()
processor = Gr00tN1d7Processor(
    modality_configs={TAG: modality_configs},
    use_percentiles=mc.use_percentiles,
    statistics=None,
    embodiment_id_mapping=None,
    image_crop_size=mc.image_crop_size,
    image_target_size=mc.image_target_size,
    random_rotation_angle=mc.random_rotation_angle,
    color_jitter_params=mc.color_jitter_params,
    model_name=mc.model_name,
    model_type=mc.backbone_model_type,
    formalize_language=mc.formalize_language,
    max_state_dim=mc.max_state_dim,
    max_action_dim=mc.max_action_dim,
    apply_sincos_state_encoding=mc.apply_sincos_state_encoding,
    max_action_horizon=mc.action_horizon,
    use_albumentations=mc.use_albumentations_transforms,
    extra_augmentation_config=mc.extra_augmentation_config,
    shortest_image_edge=mc.shortest_image_edge,
    crop_fraction=mc.crop_fraction,
    use_relative_action=mc.use_relative_action,
    exclude_state=mc.exclude_state,
    state_dropout_prob=mc.state_dropout_prob,
    use_mean_std=mc.use_mean_std,
)
assert adapter.action_horizon <= mc.action_horizon
adapter.configure_processor(processor, train_ds, TAG)  # processor.set_statistics(...), computed from train_ds

processor.train()
inputs = adapter.to_model_batch(val_ds.sample(), processor, TAG)


def tensors(x, prefix=""):
    if isinstance(x, dict) or hasattr(x, "items"):
        for k, v in x.items():
            yield from tensors(v, f"{prefix}{k}.")
    elif hasattr(x, "shape"):
        yield prefix.rstrip("."), tuple(x.shape), str(getattr(x, "dtype", ""))


for name, shape, dtype in tensors(inputs):
    print(f"{name:<32} {str(shape):<22} {dtype}")

Overriding statistics for new_embodiment


inputs.input_ids                 (4, 139)               torch.int64
inputs.attention_mask            (4, 139)               torch.int64
inputs.pixel_values              (2048, 1536)           torch.float32
inputs.image_grid_thw            (8, 3)                 torch.int64
inputs.action_mask               (4, 40, 132)           torch.float32
inputs.state                     (4, 1, 132)            torch.float32
inputs.action                    (4, 40, 132)           torch.float32
inputs.embodiment_id             (4,)                   torch.int64


## Your own dataset

Only the first cells change: pick another loader and its arguments (`LeRobotVideoDataset` with `local_path`,
`Table30v2Dataset`, ...), run the listing, edit the YAML. For example, for a LeRobot dataset whose `action` column packs an
end-effector pose and a gripper:

```yaml
modalities:
  - {path: action, kind: action, dtype: float32, shape: [7], use: true, name: action,
     split: [{name: eef, columns: [0, 6]}, {name: gripper, columns: [6, 7]}]}
gr00t:
  action:
    eef: {rep: RELATIVE, type: EEF, format: XYZ_ROTVEC, state_key: cartesian}
```

A `RELATIVE` group needs its reference state (`state_key`) to be a whole state leaf. The two ready-made files for
`finetune-groot` (`stack_tact_v4`, `viola`) are in its `configs/scratch/`.